# 02 · 변수 매핑 확정 — 신뢰·다양성 코어 + 보조 지표

> 선행: [`01-eda-2025.ipynb`](01-eda-2025.ipynb)(2코어 결정) · 방법론 근거: [`docs/groundwork/03-research-index-methodology.md`](../docs/groundwork/03-research-index-methodology.md)(Perplexity Deep Research) · 설계: [`docs/design/preprocessing-design.md`](../docs/design/preprocessing-design.md)
> SSOT 모듈: [`src/news_health_features.py`](../src/news_health_features.py) — 본 노트북은 그 매핑을 **검증·문서화**한다.

## 이 노트북이 확정하는 것
1. **신뢰 코어**(뉴스/언론 신뢰만) ↔ 일반화 신뢰(통제) ↔ 비판적 문제인식(별도) **3분리**를 α·상관·요인분석으로 입증.
2. **다양성 = Richness(폭) 주지표** 결정 — 일수가중 Shannon/Pielou를 산출하되 *본 데이터에서 균등도가 천장→변별력 낮음*을 진단(문헌 권고의 데이터 검증).
3. 검증·회피 보조 지표 가용성 재확인 → 최종 매핑표.

> ⚠️ 분석 수치는 KPF 원자료 재검증 전까지 보고서 직접 인용 금지(내부 설계 근거).

In [1]:
import sys; sys.path.insert(0, "../src"); sys.path.insert(0, "src")
import numpy as np, pandas as pd
import news_health_features as F
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, FactorAnalysis

df = F.load_2025()
pd.set_option("display.max_columns", 30)
print(f"로딩 {df.shape[0]:,}행 × {df.shape[1]}변수  ·  특수코드 {F.SPECIAL}")

로딩 6,000행 × 223변수  ·  특수코드 [9997, 9998, 9999]


## 1. 신뢰 차원 — 3분리 검증

**가설(방법론 근거)**: ① 뉴스/언론 신뢰(코어)와 ② 일반화 신뢰(직업군·사회 전반)는 분리해야 하고(Uslaner 2002; Kohring & Matthes 2007), ③ 보도 문제점 인식(Q91)은 냉소(역신뢰)인지 회의(건강한 리터러시)인지 모호하므로 신뢰에 합산하면 안 된다.

In [2]:
# 내적 일관성(Cronbach α)
for nm, cols in [("신뢰 코어(뉴스/언론)", F.TRUST_CORE),
                 ("일반화 신뢰(통제)", F.TRUST_CONTEXT),
                 ("비판적 문제인식 Q91", F.CRITICAL_PERCEPTION)]:
    X = F.mask_special(df, cols)
    print(f"{nm:18s} {len(cols):2d}문항  α={F.cronbach_alpha(X):.3f}  유효N={X.dropna().shape[0]}")

신뢰 코어(뉴스/언론)       22문항  α=0.902  유효N=6000
일반화 신뢰(통제)         10문항  α=0.799  유효N=6000
비판적 문제인식 Q91        8문항  α=0.887  유효N=6000


In [3]:
# 블록 간 상관 (코어가 일반신뢰·문제인식과 얼마나 분리되나)
core = F.mask_special(df, F.TRUST_CORE).mean(axis=1)
ctx  = F.mask_special(df, F.TRUST_CONTEXT).mean(axis=1)
crit = F.mask_special(df, F.CRITICAL_PERCEPTION).mean(axis=1)
blocks = pd.DataFrame({"신뢰코어": core, "일반화신뢰": ctx, "문제인식(Q91)": crit})
print("블록 평균 간 상관:")
print(blocks.corr().round(3).to_string())
print("\n→ 코어~일반신뢰 r≈0.53(관련되나 구분) · 코어~문제인식 r≈-0.19(거의 직교)")

블록 평균 간 상관:
            신뢰코어  일반화신뢰  문제인식(Q91)
신뢰코어       1.000  0.530     -0.194
일반화신뢰      0.530  1.000     -0.161
문제인식(Q91) -0.194 -0.161      1.000

→ 코어~일반신뢰 r≈0.53(관련되나 구분) · 코어~문제인식 r≈-0.19(거의 직교)


In [4]:
# 코어 22문항 단일차원성: 상관행렬 고유값(Kaiser 기준 >1) + 1요인 적재량
Xc = F.mask_special(df, F.TRUST_CORE).dropna()
Z = StandardScaler().fit_transform(Xc)
ev = PCA().fit(Z).explained_variance_   # 표준화 → 상관행렬 고유값
print("상위 고유값:", np.round(ev[:6], 2), " | >1 개수(Kaiser):", int((ev > 1).sum()))
print(f"제1성분 설명분산 = {ev[0]/ev.sum()*100:.1f}%")
fa = FactorAnalysis(n_components=1, random_state=0).fit(Z)
load = pd.Series(fa.components_[0], index=F.TRUST_CORE).abs()
print(f"1요인 적재량: 최소 {load.min():.2f} · 최대 {load.max():.2f} · 평균 {load.mean():.2f} (모두 동일 방향)")

상위 고유값: [7.43 1.84 1.26 1.07 0.98 0.84]  | >1 개수(Kaiser): 4
제1성분 설명분산 = 33.8%
1요인 적재량: 최소 0.32 · 최대 0.67 · 평균 0.55 (모두 동일 방향)


**해석**: 코어 α=0.90으로 내적일관성이 매우 높고, 상관행렬 **제1고유값(7.43)이 차순위(1.84)의 4배** → **강한 지배적 일반 신뢰요인**이 존재(제1성분 33.8%). 다만 고유값>1이 4개로 **하위구조(인식·역할·출처·언론인)** 도 있어, 03에서 요인점수로 하위차원을 정제할 여지가 있다(지금은 단일 신뢰점수로 충분). 일반화 신뢰는 r≈0.53로 관련되나 구분되어 **통제변수로 분리**, 문제인식(Q91)은 코어와 거의 직교(r≈−0.19)·자체 α=0.89로 별개 구성개념 → **신뢰에 합산하지 않고 '비판적 인식' 별도 지표로 보류**(냉소 vs 회의 귀속은 향후 CFA로 검증).

## 2. 다양성 차원 — 일수가중 vs Richness

01에서 **이진 이용여부의 Shannon은 count의 단조변환**(정보 없음)임을 봤다. 방법론 권고는 **이용일수(노출)를 pᵢ로 가중**하는 것. 실제로 단조변환이 깨지는지, 그리고 균등도가 변별력을 갖는지 검증한다.

In [5]:
dv = F.diversity_metrics(df)
m = dv["richness"] >= 1
print(f"뉴스 이용자(S≥1) N={int(m.sum())}  ·  매체유형 k={len(F.NEWS_DAYS)}")
print(dv[m][["richness","shannon_H","pielou_J","effective_N","hhi"]].describe().round(3).to_string())

뉴스 이용자(S≥1) N=5730  ·  매체유형 k=12
       richness  shannon_H  pielou_J  effective_N       hhi
count  5730.000   5730.000  4325.000     5730.000  5730.000
mean      3.184      0.934     0.959        3.024     0.488
std       1.825      0.618     0.052        1.679     0.309
min       1.000     -0.000     0.650        1.000     0.096
25%       2.000      0.451     0.943        1.569     0.258
50%       3.000      1.078     0.975        2.937     0.349
75%       4.000      1.379     0.995        3.972     0.722
max      12.000      2.402     1.000       11.043     1.000


In [6]:
# 핵심: 동일 Richness(S)에서 Shannon H가 변하는가? (binary면 H=ln(S)로 고정·std=0)
S, H, J = dv["richness"], dv["shannon_H"], dv["pielou_J"]
rows = []
for s in range(2, 11):
    sel = S == s
    if sel.sum() < 5: continue
    rows.append([s, int(sel.sum()), round(H[sel].mean(),3), round(H[sel].std(),3),
                 round(np.log(s),3), round(J[sel].mean(),3)])
print(pd.DataFrame(rows, columns=["S","n","H_mean","H_std","ln(S)","J_mean"]).to_string(index=False))
print(f"\nShannon H ~ Richness 상관 r={np.corrcoef(H[m],S[m])[0,1]:.3f}  (01의 binary 0.948 대비)")
print(f"Pielou J ~ Richness 상관 r={np.corrcoef(J[J.notna()],S[J.notna()])[0,1]:.3f}  (≈0 → 균등도는 폭과 독립)")

 S    n  H_mean  H_std  ln(S)  J_mean
 2  658   0.661  0.058  0.693   0.954
 3 1442   1.055  0.057  1.099   0.960
 4  974   1.330  0.058  1.386   0.959
 5  686   1.547  0.054  1.609   0.961
 6  314   1.724  0.058  1.792   0.962
 7  135   1.870  0.060  1.946   0.961
 8   55   1.997  0.074  2.079   0.960
 9   28   2.104  0.071  2.197   0.958
10   21   2.196  0.059  2.303   0.954

Shannon H ~ Richness 상관 r=0.943  (01의 binary 0.948 대비)
Pielou J ~ Richness 상관 r=0.020  (≈0 → 균등도는 폭과 독립)


**해석**: 일수가중으로 단조변환은 **기술적으로 깨진다**(동일 S에서 H_std>0, J⊥S). 그러나 **Pielou J≈0.96로 천장에 몰려 변별력이 거의 없다** — 주간 이용일수(0~5)가 coarse해 다매체 이용자는 대체로 고르게 소비. 따라서 본 데이터의 실질 다양성 신호는 **Richness(폭)** 이다.

→ **결정**: 다양성 주지표 = **Richness(뉴스 매체유형 수)**. 일수가중 Shannon/Pielou/HHI는 *문헌 권고 이행 + 강건성 진단*으로 병행 보고(균등도 천장 사실을 명시). 맹목 적용이 아니라 **검증 후 채택**.

## 3. 보조 지표 — 검증습관·회피 (재확인)

In [7]:
vp = F.mask_special(df, F.VERIFY_PROXY)
print("검증 프록시 유효%(구조적 결측):")
print((100*vp.notna().mean()).round(1).to_string())
print(f"\n회피 플래그(Q84==9998) = {100*F.avoid_flag(df).mean():.2f}%  → 코어 아님, 세그먼트 태그용")

검증 프록시 유효%(구조적 결측):
Q93      69.9
Q34_3    64.5
Q56_2    26.3
Q34_5    64.5
Q56_4    26.3

회피 플래그(Q84==9998) = 4.35%  → 코어 아님, 세그먼트 태그용


## 4. 최종 변수 매핑표 (SSOT 확정)

| 역할 | 변수 | 처리 | 비고 |
|------|------|------|------|
| **신뢰 코어** | Q85·Q86·Q87·Q88·Q94 + Q95_5(언론인) | 특수코드 NaN → z-표준화 → (요인)합성 | α=0.90 · 지배적 일반요인(+하위구조) |
| 일반화 신뢰(통제) | Q95(언론인 제외 9개)·Q96 | 코어 제외, 맥락/통제 | r≈0.53 |
| 비판적 인식(별도) | Q91_1~8 | 신뢰 합산 금지, 별도 지표 | 냉소/회의 CFA 보류 |
| **다양성 코어** | NEWS_DAYS 12유형 → **Richness** | 일수>0 카운트(주), Shannon/Pielou(보조) | 균등도 천장 |
| 검증습관(보조) | Q93·Q34_3·Q56_2·Q34_5·Q56_4 | 응답자 한정 부가 | 결측 30~74% |
| 회피(플래그) | Q84==9998 | 이진 세그먼트 태그 | 4.35% |
| 가중·연령 | WT, DQ3→연령대 비닝 | 모든 모집단 추정 가중 | EDA P3·P4 |

### 다음(03-health-index)
1. 신뢰: z-표준화 → (동일가중 기본 / PCA가중 민감도) → 0~100 척도화.
2. 다양성: Richness Min-Max(0~k) 정규화 → 신뢰와 **기하평균** 집계(JRC 권고, 보완성↓).
3. 강건성: 동일가중 vs PCA가중, 산술 vs 기하 집계 민감도표. 검증·회피는 보조 레이어.